In [2]:
# 只需執行一次：把原始 CSV 轉成 Parquet（已完成，勿重跑）
# Parquet 是 columnar 格式，之後的 query 讀取速度比 .csv.gz 快很多

import duckdb

duckdb.sql("""
COPY (SELECT * FROM read_csv_auto('../data/raw/2019-Oct.csv.gz')) -- read_csv_auto：讀取 CSV，自動判斷欄位名稱和資料型態，可直接讀 .gz 壓縮檔
TO '../data/processed/events_2019_10.parquet' (FORMAT PARQUET) -- COPY (...) TO：把 query 結果寫成檔案
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [3]:
path = "'../data/processed/events_2019_10.parquet'"
# 把檔案路徑存進變數 path。外層雙引號是 Python 字串，
# 內層單引號會一起放進 SQL，因為 SQL 裡的檔名要用單引號包住

## data profiling

### 1. 欄位名稱與資料型態

In [12]:
duckdb.sql(f"DESCRIBE SELECT * FROM {path}").show()
#DESCRIBE 的 null = YES：這個欄位允許存 NULL（規則）

┌───────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│  column_name  │ column_type │  null   │   key   │ default │  extra  │
│    varchar    │   varchar   │ varchar │ varchar │ varchar │ varchar │
├───────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ event_time    │ TIMESTAMP   │ YES     │ NULL    │ NULL    │ NULL    │
│ event_type    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ product_id    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ category_id   │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ category_code │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ brand         │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ price         │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ user_id       │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ user_session  │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
└───────────────┴─────────────┴─────────┴─────────┴─────────┴───

### 2. 前 10 筆資料，看實際長相

In [ ]:
duckdb.sql(f"SELECT * FROM {path} LIMIT 10").show()

┌─────────────────────┬────────────┬────────────┬─────────────────────┬─────────────────────────────────────┬──────────┬─────────┬───────────┬──────────────────────────────────────┐
│     event_time      │ event_type │ product_id │     category_id     │            category_code            │  brand   │  price  │  user_id  │             user_session             │
│      timestamp      │  varchar   │   int64    │        int64        │               varchar               │ varchar  │ double  │   int64   │               varchar                │
├─────────────────────┼────────────┼────────────┼─────────────────────┼─────────────────────────────────────┼──────────┼─────────┼───────────┼──────────────────────────────────────┤
│ 2019-10-01 00:00:00 │ view       │   44600062 │ 2103807459595387724 │ NULL                                │ shiseido │   35.79 │ 541312140 │ 72d76fde-8bb3-4e00-8c23-a032dfed738c │
│ 2019-10-01 00:00:00 │ view       │    3900821 │ 2053013552326770905 │ appliances.environ

### 3. 每個欄位的統計摘要：min、max、不重複值數量、平均、missing 比例等

In [ ]:
duckdb.sql(f"SUMMARIZE SELECT * FROM {path}").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌───────────────┬─────────────┬──────────────────────────────────────┬──────────────────────────────────────┬───────────────┬────────────────────────────┬────────────────────────┬────────────────────────────┬────────────────────────────┬────────────────────────────┬──────────┬─────────────────┐
│  column_name  │ column_type │                 min                  │                 max                  │ approx_unique │            avg             │          std           │            q25             │            q50             │            q75             │  count   │ null_percentage │
│    varchar    │   varchar   │               varchar                │               varchar                │     int64     │          varchar           │        varchar         │          varchar           │          varchar           │          varchar           │  int64   │  decimal(9,2)   │
├───────────────┼─────────────┼──────────────────────────────────────┼──────────────────────────────────────┼───

### Query 1：資料規模

In [ ]:
# 呼叫 DuckDB 執行 SQL；f-string 讓 {path} 替換成變數值
duckdb.sql(f"""
SELECT COUNT(*) AS n_events,                        -- 總列數
       COUNT(DISTINCT user_id) AS n_users,          -- 不重複 user 數
       COUNT(DISTINCT user_session) AS n_sessions,  -- 不重複 session 數
       MIN(event_time), MAX(event_time)             -- 最早和最晚時間
FROM {path}
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────┬─────────┬────────────┬─────────────────────┬─────────────────────┐
│ n_events │ n_users │ n_sessions │   min(event_time)   │   max(event_time)   │
│  int64   │  int64  │   int64    │      timestamp      │      timestamp      │
├──────────┼─────────┼────────────┼─────────────────────┼─────────────────────┤
│ 42448764 │ 3022290 │    9244421 │ 2019-10-01 00:00:00 │ 2019-10-31 23:59:59 │
└──────────┴─────────┴────────────┴─────────────────────┴─────────────────────┘



### Query 2：各類 event 的數量和佔比

In [ ]:
duckdb.sql(f"""
SELECT event_type, COUNT(*) AS n,           -- event 類型，以及每一類的筆數
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct
       -- 每類筆數 ÷ 全部筆數 × 100，四捨五入到小數 2 位
       -- SUM(COUNT(*)) OVER ()：把各組的 COUNT(*) 加總，得到總筆數
FROM {path}
GROUP BY event_type                         -- 依 event_type 分組
ORDER BY n DESC                             -- 依筆數由大到小排序
""").show()

┌────────────┬──────────┬────────┐
│ event_type │    n     │  pct   │
│  varchar   │  int64   │ double │
├────────────┼──────────┼────────┤
│ view       │ 40779399 │  96.07 │
│ cart       │   926516 │   2.18 │
│ purchase   │   742849 │   1.75 │
└────────────┴──────────┴────────┘



### 驗證：一個 session 通常包含多筆 event

In [10]:
duckdb.sql(f"""
WITH s AS (
    SELECT user_session, COUNT(*) AS n_events   -- 每個 session 的 event 數
    FROM {path}
    GROUP BY user_session
)
SELECT COUNT(*) AS n_sessions,
       SUM(CASE WHEN n_events = 1 THEN 1 ELSE 0 END) AS n_single_event_sessions,  -- 只有 1 筆 event 的 session
       ROUND(AVG(n_events), 2) AS avg_events,
       MEDIAN(n_events) AS median_events,
       MAX(n_events) AS max_events
FROM s
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬─────────────────────────┬────────────┬───────────────┬────────────┐
│ n_sessions │ n_single_event_sessions │ avg_events │ median_events │ max_events │
│   int64    │         int128          │   double   │    double     │   int64    │
├────────────┼─────────────────────────┼────────────┼───────────────┼────────────┤
│    9244422 │                 3269577 │       4.59 │           2.0 │       1159 │
└────────────┴─────────────────────────┴────────────┴───────────────┴────────────┘



### Query 3：Missing 比例

In [ ]:
duckdb.sql(f"""
SELECT ROUND(100.0 * AVG(CASE WHEN category_code IS NULL THEN 1 ELSE 0 END), 2) AS pct_missing_category,
       -- CASE WHEN：如果 category_code 是空值就給 1，否則給 0
       -- 對這串 0/1 取平均 = 空值的比例，× 100 變成百分比
       ROUND(100.0 * AVG(CASE WHEN brand IS NULL THEN 1 ELSE 0 END), 2) AS pct_missing_brand,
       -- 同樣的邏輯，換成 brand 欄位
       ROUND(100.0 * AVG(CASE WHEN user_session IS NULL THEN 1 ELSE 0 END), 4) AS pct_missing_session
       -- 同樣的邏輯，換成 user_session；取到小數 4 位，因為預期空值極少
FROM {path}
""").show()

┌──────────────────────┬───────────────────┬─────────────────────┐
│ pct_missing_category │ pct_missing_brand │ pct_missing_session │
│        double        │      double       │       double        │
├──────────────────────┼───────────────────┼─────────────────────┤
│                31.84 │              14.4 │                 0.0 │
└──────────────────────┴───────────────────┴─────────────────────┘



### 驗證所有欄位的缺值數

In [11]:
# 每個欄位的 NULL 數量
# COUNT(欄位) 只計算非 NULL 的列，所以 COUNT(*) - COUNT(欄位) = NULL 數
duckdb.sql(f"""
SELECT COUNT(*) - COUNT(event_time)    AS null_event_time,
       COUNT(*) - COUNT(event_type)    AS null_event_type,
       COUNT(*) - COUNT(product_id)    AS null_product_id,
       COUNT(*) - COUNT(category_id)   AS null_category_id,
       COUNT(*) - COUNT(category_code) AS null_category_code,
       COUNT(*) - COUNT(brand)         AS null_brand,
       COUNT(*) - COUNT(price)         AS null_price,
       COUNT(*) - COUNT(user_id)       AS null_user_id,
       COUNT(*) - COUNT(user_session)  AS null_user_session
FROM {path}
""").show()

┌─────────────────┬─────────────────┬─────────────────┬──────────────────┬────────────────────┬────────────┬────────────┬──────────────┬───────────────────┐
│ null_event_time │ null_event_type │ null_product_id │ null_category_id │ null_category_code │ null_brand │ null_price │ null_user_id │ null_user_session │
│      int64      │      int64      │      int64      │      int64       │       int64        │   int64    │   int64    │    int64     │       int64       │
├─────────────────┼─────────────────┼─────────────────┼──────────────────┼────────────────────┼────────────┼────────────┼──────────────┼───────────────────┤
│               0 │               0 │               0 │                0 │           13515609 │    6113008 │          0 │            0 │                 2 │
└─────────────────┴─────────────────┴─────────────────┴──────────────────┴────────────────────┴────────────┴────────────┴──────────────┴───────────────────┘



找 Missing 比例的目的是決定每個欄位能不能用、要怎麼處理。 Missing 的比例和原因，會直接影響後面的分析和 model。具體有四個用途：

1. 判斷欄位還能不能用
缺很多（例如 category_code 缺 31.8%）就不能當主要分析維度，否則按類別分析時會有將近三分之一的資料落在「NULL」那組，結論會失真。所以我們改用沒有缺值的 category_id。

2. 決定處理方式
刪除有缺值的列、填補（例如填平均數或 'unknown'）、或保留缺值讓 model 自己處理（LightGBM 可以直接處理 NULL）。比例決定選哪個：缺 14.4% 的 brand 如果整列刪掉，會損失 600 多萬筆資料，所以改成填 'unknown'。

3. 檢查缺值是不是隨機的
這是最重要的一點。如果缺值集中在特定情況，就不是隨機的，處理不當會造成偏差。例如：brand 缺值是否集中在便宜的商品或某些類別？如果是，「沒有品牌」本身就帶有資訊，可以做成一個 feature（is_brand_missing）。

4. 寫進 README 的 limitation
Reviewer 會想知道你的資料有哪些缺陷、你怎麼處理。例如「category_code 缺 31.8%，因此類別分析改用 category_id」，這一句就展現了你有檢查資料品質，而不是拿到資料就直接跑 model。

In [7]:
# Session 層級：有沒有 cart、有沒有 purchase 的交叉表
duckdb.sql(f"""
WITH s AS (
-- 第一段（CTE）：把 event 層級壓成 session 層級，每個 session 一列
    SELECT user_session,
           MAX(CASE WHEN event_type = 'cart' THEN 1 ELSE 0 END) AS has_cart, -- 每筆 event 是 cart 就給 1，否則 0；
           -- MAX 取組內最大值：只要出現過至少一次 cart 就是 1
           MAX(CASE WHEN event_type = 'purchase' THEN 1 ELSE 0 END) AS has_purchase
    FROM {path}
    GROUP BY user_session  -- 同一個 session 的所有 event 收成一組
)

-- 第二段：對 session 表依 (has_cart, has_purchase) 的四種組合計數
--   0,0 只瀏覽 / 0,1 沒加購物車就購買 / 1,0 加了沒買 / 1,1 加了也買
SELECT has_cart, has_purchase, COUNT(*) AS n_sessions,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct
FROM s
GROUP BY has_cart, has_purchase
ORDER BY has_cart, has_purchase
""").show()

# 完全重複的列有幾筆
duckdb.sql(f"""
SELECT (SELECT COUNT(*) FROM {path})
     - (SELECT COUNT(*) FROM (SELECT DISTINCT * FROM {path})) AS n_duplicate_rows
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────┬──────────────┬────────────┬────────┐
│ has_cart │ has_purchase │ n_sessions │  pct   │
│  int32   │    int32     │   int64    │ double │
├──────────┼──────────────┼────────────┼────────┤
│        0 │            0 │    8333625 │  90.15 │
│        0 │            1 │     337699 │   3.65 │
│        1 │            0 │     281237 │   3.04 │
│        1 │            1 │     291861 │   3.16 │
└──────────┴──────────────┴────────────┴────────┘



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────────┐
│ n_duplicate_rows │
│      int64       │
├──────────────────┤
│            30220 │
└──────────────────┘



### 各 event_type 的重複列數 & 重複比例

In [ ]:
duckdb.sql(f"""
WITH d AS (
    SELECT event_type,
           COUNT(*) AS n_rows,
           COUNT(*) - COUNT(DISTINCT (event_time, event_type, product_id, user_id, user_session, price)) AS n_duplicates
    FROM {path}
    GROUP BY event_type
)
SELECT event_type,
       n_rows,
       n_duplicates,
       ROUND(100.0 * n_duplicates / n_rows, 3) AS pct_duplicates   -- 取到小數 3 位，view 的比例很小
FROM d
ORDER BY pct_duplicates DESC
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬──────────┬──────────────┬────────────────┐
│ event_type │  n_rows  │ n_duplicates │ pct_duplicates │
│  varchar   │  int64   │    int64     │     double     │
├────────────┼──────────┼──────────────┼────────────────┤
│ cart       │   926516 │        28073 │           3.03 │
│ purchase   │   742849 │           76 │           0.01 │
│ view       │ 40779399 │         2071 │          0.005 │
└────────────┴──────────┴──────────────┴────────────────┘



重複幾乎全部集中在 cart。View 和 purchase 的重複少到可以忽略，不是整個 tracking 系統都有問題，而是 cart 這個動作特有的現象。最可能的原因是使用者在同一秒內連點「加入購物車」，或是加入多件同樣商品時被記成多筆。兩者在這份資料裡無法區分，但都不是嚴重的資料錯誤。
### 處理決定
不刪除任何資料：重複比例低，且有合理的使用者行為解釋
Label（has_cart、has_purchase）用 MAX 計算，不受重複影響
之後做 cart 相關的計數 feature 時，用 COUNT(DISTINCT product_id)，避免連點造成的灌水
README 的 data quality 段落記一句：cart events 有 3% 為同秒完全重複，判斷為連點或多件加入，未刪除，計數 feature 改用 distinct count

### Session 層級的 cart / purchase 交叉表
目的：確認有多少 purchase 沒有經過 cart，用來決定 label 的定義

In [14]:
# 把 event 層級壓成 session 層級，存成一張名為 sessions 的表（一列 = 一個 session）
duckdb.sql(f"""
CREATE OR REPLACE TABLE sessions AS
SELECT user_session,
       MAX(CASE WHEN event_type = 'cart' THEN 1 ELSE 0 END) AS has_cart,
       MAX(CASE WHEN event_type = 'purchase' THEN 1 ELSE 0 END) AS has_purchase
FROM {path}
GROUP BY user_session
""")

# 看前 10 列
duckdb.sql("SELECT * FROM sessions LIMIT 10").show()

# 確認列數，應該是 9,244,422
duckdb.sql("SELECT COUNT(*) AS n_rows FROM sessions").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────────────────────────────┬──────────┬──────────────┐
│             user_session             │ has_cart │ has_purchase │
│               varchar                │  int32   │    int32     │
├──────────────────────────────────────┼──────────┼──────────────┤
│ 4d3ea6b8-f816-421e-9717-3d858dac273d │        0 │            0 │
│ e36b819b-95de-427a-999d-f2fc9c9055fb │        0 │            0 │
│ 0150ec9f-c1cf-4539-971c-a2abf8edc97c │        0 │            0 │
│ ab830339-5f46-4fe6-a68f-bb4d35ce1080 │        0 │            0 │
│ 4ef094aa-6dcb-47a0-90f0-2e687d898c28 │        0 │            0 │
│ 9ca847cc-64c6-4848-99c9-2b842d59b1f3 │        0 │            0 │
│ 11c4be07-4843-4d5f-b222-f5a05db77d7d │        0 │            0 │
│ e32db82f-f4c1-4c2e-9543-27befc650562 │        0 │            0 │
│ 3aa9d7b2-f773-41ce-9b88-eb9c62606070 │        0 │            0 │
│ 83269baf-daf9-474d-8754-ca84b7dc16ea │        0 │            0 │
└──────────────────────────────────────┴──────────┴───────────

In [15]:
# 四種組合各有幾個 session
duckdb.sql("""
SELECT has_cart,
       has_purchase,
       COUNT(*) AS n_sessions,
       -- 佔全部 session 的百分比
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct_of_all,
       -- 在同樣 has_cart 的 session 裡的百分比
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY has_cart), 2) AS pct_within_cart_group
FROM sessions
GROUP BY has_cart, has_purchase
ORDER BY has_cart, has_purchase
""").show()

┌──────────┬──────────────┬────────────┬────────────┬───────────────────────┐
│ has_cart │ has_purchase │ n_sessions │ pct_of_all │ pct_within_cart_group │
│  int32   │    int32     │   int64    │   double   │        double         │
├──────────┼──────────────┼────────────┼────────────┼───────────────────────┤
│        0 │            0 │    8333625 │      90.15 │                 96.11 │
│        0 │            1 │     337699 │       3.65 │                  3.89 │
│        1 │            0 │     281237 │       3.04 │                 49.07 │
│        1 │            1 │     291861 │       3.16 │                 50.93 │
└──────────┴──────────────┴────────────┴────────────┴───────────────────────┘



`pct_within_cart_group`  前兩列：沒有加入購物車的 session
- 96.11% 沒有購買
- 3.89% 沒加購物車就直接購買

`pct_within_cart_group`  後兩列：有加入購物車的 session
- 49.07% 加了沒買，也就是 session-level 的 **cart abandonment rate**
- 50.93% 加了也買了，也就是 **cart conversion rate** 

### Session 交叉表發現
- 有 cart 的 session 共 573,098 個，其中 **50.93%** 結帳（cart conversion rate），49.07% 放棄
- 有購買的 session 共 629,560 個，其中 **53.6%** 沒有 cart event，代表超過一半的購買不經過購物車
- **Scope**：本 project 只預測「加入購物車後是否結帳」，直接購買的 session 不在範圍內

## 建立 label 表

對每個有加入購物車的 session，回答一個問題：**第一次加入購物車之後，有沒有購買？**\
有就是 `label = 1`，沒有就是 `label = 0`。

要回答這個問題需要兩個資訊：

1. 第一次加入購物車是什麼時候
2. 那個時間之後有沒有 `purchase`

### Step 1：找出每個 session 第一次 cart 的時間

In [17]:
# 每個有 cart 的 session，第一次 cart 是什麼時候
duckdb.sql(f"""
CREATE OR REPLACE TABLE first_cart AS
SELECT user_session,
       ANY_VALUE(user_id) AS user_id,        -- 同一個 session 屬於同一個 user，取任一值即可
       MIN(event_time) AS first_cart_time    -- 最早的 cart 時間
FROM {path}
WHERE event_type = 'cart'                    -- 只看 cart 事件
GROUP BY user_session                        -- 每個 session 一列
""")

duckdb.sql("SELECT * FROM first_cart LIMIT 5").show()
duckdb.sql("SELECT COUNT(*) AS n FROM first_cart").show()

┌──────────────────────────────────────┬───────────┬─────────────────────┐
│             user_session             │  user_id  │   first_cart_time   │
│               varchar                │   int64   │      timestamp      │
├──────────────────────────────────────┼───────────┼─────────────────────┤
│ 161c09ac-1237-447f-ac74-2152e9f37518 │ 550373867 │ 2019-10-04 11:09:27 │
│ cca5930e-1af1-4600-a05d-f361437a624b │ 551759115 │ 2019-10-04 11:11:27 │
│ 415d357f-6233-414f-a6ad-341933b5a757 │ 514328919 │ 2019-10-04 11:13:32 │
│ 793f03ed-3a22-4652-85da-012c575c1bfa │ 515321620 │ 2019-10-04 11:17:30 │
│ fbe4e5dd-1c75-4731-b8a7-a09a3b7dea86 │ 519125265 │ 2019-10-04 11:17:37 │
└──────────────────────────────────────┴───────────┴─────────────────────┘

┌────────┐
│   n    │
│ int64  │
├────────┤
│ 573098 │
└────────┘



### Step 2：拿一個 session 看 JOIN 在做什麼

`first_cart` 只有「第一次 cart 的時間」，但要知道之後有沒有買，必須把這個 session 的**所有 event** 接回來比對。這就是 `JOIN`。

先只拿一個 session 看：

In [19]:
# 挑一個有 cart 也有 purchase 的 session，看它每筆 event 跟第一次 cart 時間的關係
duckdb.sql(f"""
SELECT f.user_session,
       e.event_time,
       e.event_type,
       e.product_id,
       f.first_cart_time,
       e.event_time >= f.first_cart_time AS after_first_cart   -- 這筆 event 是否發生在第一次 cart 之後
FROM first_cart f
JOIN {path} e
  ON e.user_session = f.user_session          -- 用 session 把兩張表接起來
WHERE f.user_session = (
    SELECT user_session FROM sessions
    WHERE has_cart = 1 AND has_purchase = 1
    LIMIT 1
)
ORDER BY e.event_time
""").show()

┌──────────────────────────────────────┬─────────────────────┬────────────┬────────────┬─────────────────────┬──────────────────┐
│             user_session             │     event_time      │ event_type │ product_id │   first_cart_time   │ after_first_cart │
│               varchar                │      timestamp      │  varchar   │   int64    │      timestamp      │     boolean      │
├──────────────────────────────────────┼─────────────────────┼────────────┼────────────┼─────────────────────┼──────────────────┤
│ dc63f75f-075b-44c6-b62a-77653e885a3f │ 2019-10-01 07:37:06 │ view       │    1005135 │ 2019-10-01 07:38:01 │ false            │
│ dc63f75f-075b-44c6-b62a-77653e885a3f │ 2019-10-01 07:37:25 │ view       │    1004238 │ 2019-10-01 07:38:01 │ false            │
│ dc63f75f-075b-44c6-b62a-77653e885a3f │ 2019-10-01 07:38:01 │ cart       │    1004238 │ 2019-10-01 07:38:01 │ true             │
│ dc63f75f-075b-44c6-b62a-77653e885a3f │ 2019-10-01 07:38:40 │ purchase   │    1004238 │ 2

### Step 3：對全部 session 做同樣的事，壓成一列一個 label

Step 2 是**一個 session** 的展開版本。現在對 **57 萬個 session** 都做，然後每個 session 壓回一列：只要有任何一筆是「cart 之後的 `purchase`」，`label` 就是 `1`。

In [20]:
# 一列 = 一個有 cart 的 session，label = 第一次 cart 之後是否有購買
duckdb.sql(f"""
CREATE OR REPLACE TABLE cart_sessions AS
SELECT f.user_session,
       f.user_id,
       f.first_cart_time,
       MAX(CASE WHEN e.event_type = 'purchase'
                 AND e.event_time >= f.first_cart_time
                THEN 1 ELSE 0 END) AS label
FROM first_cart f
JOIN {path} e
  ON e.user_session = f.user_session
GROUP BY f.user_session, f.user_id, f.first_cart_time
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

### Step 4：檢查結果

In [22]:
duckdb.sql("SELECT * FROM cart_sessions LIMIT 5").show()

duckdb.sql("""
SELECT label, COUNT(*) AS n,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct
FROM cart_sessions
GROUP BY label
""").show()

┌──────────────────────────────────────┬───────────┬─────────────────────┬───────┐
│             user_session             │  user_id  │   first_cart_time   │ label │
│               varchar                │   int64   │      timestamp      │ int32 │
├──────────────────────────────────────┼───────────┼─────────────────────┼───────┤
│ 368dbeb6-144e-4c00-85f6-8e3fdb51b7df │ 523284146 │ 2019-10-01 10:17:27 │     1 │
│ bd8b7334-1930-4acd-a658-443a89a44167 │ 544980203 │ 2019-10-01 16:47:44 │     0 │
│ d71ae9d8-4d9f-419f-8b2e-60c4bd81884e │ 519752115 │ 2019-10-02 16:01:17 │     1 │
│ 35df4425-b865-488a-8d06-d8229268e7e2 │ 512840791 │ 2019-10-02 16:17:40 │     0 │
│ 2b05e78b-bae5-493e-a9aa-7d4f3335930e │ 541545208 │ 2019-10-02 15:37:22 │     1 │
└──────────────────────────────────────┴───────────┴─────────────────────┴───────┘

┌───────┬────────┬────────┐
│ label │   n    │  pct   │
│ int32 │ int64  │ double │
├───────┼────────┼────────┤
│     0 │ 282868 │  49.36 │
│     1 │ 290229 │  50.64 │
└

### Label 表 `cart_sessions`
- 一列 = 一個有 cart 的 session，共 573,097 個（1 個 NULL session 在 JOIN 時排除）
- Label = 1：第一次 cart 之後有購買，290,229 個（50.64%）
- 1,632 個 session 的購買只發生在第一次 cart 之前，label = 0
- Label 平衡，不需處理 class imbalance

### 第一批 features — cart 之前的行為

In [23]:
# 每個 cart session 在第一次 cart 之前的行為
duckdb.sql(f"""
CREATE OR REPLACE TABLE feat_pre_cart AS
SELECT f.user_session,
       -- cart 前瀏覽了幾次
       COUNT(e.event_type) FILTER (WHERE e.event_type = 'view') AS n_views_before_cart,
       -- cart 前瀏覽了幾個不同商品
       COUNT(DISTINCT e.product_id) FILTER (WHERE e.event_type = 'view') AS n_products_viewed_before_cart,
       -- 從 session 開始到第一次 cart 經過幾秒
       COALESCE(DATE_DIFF('second', MIN(e.event_time), f.first_cart_time), 0) AS secs_to_first_cart,
       -- 時間 features
       HOUR(f.first_cart_time) AS cart_hour,
       DAYOFWEEK(f.first_cart_time) AS cart_dow
FROM first_cart f
LEFT JOIN {path} e
  ON e.user_session = f.user_session
 AND e.event_time < f.first_cart_time        -- 關鍵：只接 cart 之前的 event
GROUP BY f.user_session, f.first_cart_time
""")

duckdb.sql("SELECT * FROM feat_pre_cart LIMIT 5").show()
duckdb.sql("SELECT COUNT(*) FROM feat_pre_cart").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────────────────────────────┬─────────────────────┬───────────────────────────────┬────────────────────┬───────────┬──────────┐
│             user_session             │ n_views_before_cart │ n_products_viewed_before_cart │ secs_to_first_cart │ cart_hour │ cart_dow │
│               varchar                │        int64        │             int64             │       int64        │   int64   │  int64   │
├──────────────────────────────────────┼─────────────────────┼───────────────────────────────┼────────────────────┼───────────┼──────────┤
│ 03c939ed-ebcf-4f5b-8be3-e0c3b26d46e8 │                  20 │                             9 │               1289 │        15 │        2 │
│ 3ecc1eaa-3a04-4332-875f-0ad9ddd1bb94 │                   3 │                             2 │                172 │        15 │        2 │
│ af57538e-a721-48dd-af40-781c97006a0c │                   2 │                             1 │                 51 │        15 │        2 │
│ b49bac74-6367-402a-b2a4-f

### 第二批 features — 被加入購物車的商品

In [24]:
# 第一次 cart 的那個商品的屬性
duckdb.sql(f"""
CREATE OR REPLACE TABLE feat_cart_product AS
SELECT f.user_session,
       ANY_VALUE(e.product_id) AS cart_product_id,
       ANY_VALUE(e.price) AS cart_price,
       ANY_VALUE(e.category_id) AS cart_category_id,
       ANY_VALUE(CASE WHEN e.brand IS NULL THEN 1 ELSE 0 END) AS cart_brand_missing
FROM first_cart f
JOIN {path} e
  ON e.user_session = f.user_session
 AND e.event_type = 'cart'
 AND e.event_time = f.first_cart_time     -- 只接第一次 cart 的那筆 event
GROUP BY f.user_session
""")

duckdb.sql("SELECT * FROM feat_cart_product LIMIT 5").show()
duckdb.sql("SELECT COUNT(*) FROM feat_cart_product").show()

┌──────────────────────────────────────┬─────────────────┬────────────┬─────────────────────┬────────────────────┐
│             user_session             │ cart_product_id │ cart_price │  cart_category_id   │ cart_brand_missing │
│               varchar                │      int64      │   double   │        int64        │       int32        │
├──────────────────────────────────────┼─────────────────┼────────────┼─────────────────────┼────────────────────┤
│ c29b77f3-0e11-4f5c-8639-a8eb4b34b616 │         1005115 │     975.56 │ 2053013555631882655 │                  0 │
│ b68c19be-3549-4632-bd74-778ad186e378 │        12703300 │      64.35 │ 2053013553559896355 │                  0 │
│ 828b8a5f-7a4e-42c5-9dba-a435b4233769 │         1004873 │     388.43 │ 2053013555631882655 │                  0 │
│ 576d2c07-6602-4635-b205-9c8cd89dedf8 │         1004835 │     230.89 │ 2053013555631882655 │                  0 │
│ e1f861d9-4776-4247-bb18-2a3cc2977de9 │         1004961 │     179.35 │ 20530135

### 組合 model table

把 label 和兩批 features 組合成一張建模用的表，切分 train / test，存成檔案。之後就可以開始建 model。
- **三張表一起 `JOIN`**：`c`、`p`、`cp` 分別是三張表的簡稱。三張表都是一列一個 session，用 `user_session` 接起來後仍然是一列一個 session，只是欄位變多了。因為 `cart_sessions` 已經排除了 `NULL` session，結果應該是 **573,097 列**。
- **`AVG(label)`**：`label` 只有 `0` 和 `1`，取平均就是 `1` 的比例，跟之前算 missing 比例的技巧一樣。
- **Time-based split**：用時間切而不是隨機切。Model 實際上線是用過去的資料預測未來，所以 test set 也要是「比 train 更晚」的資料，才能模擬真實情況。隨機切會讓 model 在訓練時看到未來的模式，評估結果會偏樂觀。
  - 10/1 到 10/24 約佔四分之三，當 **train**
  - 10/25 到 10/31 當 **test**

In [25]:
# 把 label 和兩批 features 依 session 接成一張表
duckdb.sql("""
CREATE OR REPLACE TABLE model_table AS
SELECT c.user_session,
       c.user_id,
       c.first_cart_time,
       c.label,
       -- cart 前的行為
       p.n_views_before_cart,
       p.n_products_viewed_before_cart,
       p.secs_to_first_cart,
       p.cart_hour,
       p.cart_dow,
       -- 被加入購物車的商品
       cp.cart_product_id,
       cp.cart_price,
       cp.cart_category_id,
       cp.cart_brand_missing,
       -- time-based split：10/25 之前為 train，之後為 test
       CASE WHEN c.first_cart_time < TIMESTAMP '2019-10-25' THEN 'train' ELSE 'test' END AS split
FROM cart_sessions c
JOIN feat_pre_cart p      ON p.user_session = c.user_session
JOIN feat_cart_product cp ON cp.user_session = c.user_session
""")

duckdb.sql("SELECT COUNT(*) AS n FROM model_table").show()

# 每個 split 的筆數和購買比例
duckdb.sql("""
SELECT split,
       COUNT(*) AS n,
       ROUND(100.0 * AVG(label), 2) AS pct_label_1
FROM model_table
GROUP BY split
""").show()

┌────────┐
│   n    │
│ int64  │
├────────┤
│ 573097 │
└────────┘

┌─────────┬────────┬─────────────┐
│  split  │   n    │ pct_label_1 │
│ varchar │ int64  │   double    │
├─────────┼────────┼─────────────┤
│ train   │ 463222 │       50.32 │
│ test    │ 109875 │       51.98 │
└─────────┴────────┴─────────────┘



In [26]:
duckdb.sql("""
COPY model_table TO '../data/processed/model_table.parquet' (FORMAT PARQUET)
""")

In [4]:
# 使用者在這次第一次 cart 之前、其他 session 裡的行為
duckdb.sql(f"""
CREATE OR REPLACE TABLE feat_user_history AS
SELECT m.user_session,
       COUNT(e.event_type) FILTER (WHERE e.event_type = 'purchase') AS prior_purchases,
       COUNT(e.event_type) FILTER (WHERE e.event_type = 'cart')     AS prior_carts,
       COUNT(DISTINCT e.user_session)                               AS prior_sessions
FROM '../data/processed/model_table.parquet' m
LEFT JOIN {path} e
  ON e.user_id = m.user_id                    -- 同一個使用者
 AND e.event_time < m.first_cart_time         -- 只看這次 cart 之前，防止 leakage
 AND e.user_session <> m.user_session         -- 排除這次 session（已經在 cart 前 features 裡）
GROUP BY m.user_session
""")

duckdb.sql("SELECT * FROM feat_user_history LIMIT 5").show()
duckdb.sql("SELECT COUNT(*) FROM feat_user_history").show()

duckdb.sql("""
COPY feat_user_history TO '../data/processed/feat_user_history.parquet' (FORMAT PARQUET)
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────────────────────────────┬─────────────────┬─────────────┬────────────────┐
│             user_session             │ prior_purchases │ prior_carts │ prior_sessions │
│               varchar                │      int64      │    int64    │     int64      │
├──────────────────────────────────────┼─────────────────┼─────────────┼────────────────┤
│ e2c8b3f1-e600-4949-adba-b2a3013fb9fa │              15 │          10 │             12 │
│ 0230422e-c469-4b00-85a5-429aa5d7472f │               0 │           0 │              3 │
│ 152488bf-6db6-4d5f-9aac-6d992d54d6b4 │               2 │           0 │             15 │
│ 33b230c0-28ed-4429-99ec-d46746572deb │               0 │           0 │              2 │
│ 0cf08d5d-be68-491d-8856-62cc0de51f60 │               1 │           3 │             46 │
└──────────────────────────────────────┴─────────────────┴─────────────┴────────────────┘

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│       573097 │
└─────────────